In [1]:
import pandas as pd

df = pd.read_csv('khk.noun.tsv', sep = '\t')
df

,prompt,answer
0,багтраагаа,багтраа <voc>
1,дэнсээс,дэнс <abl>
2,тархалтыг,тархалт <acc>
3,дуулийг,дууль <acc>
4,гуалингаар,гуалин <ins>
...,...,...
14391,үтрээний,үтрээ <gen>
14392,лавлагаанд,лавлагаа <dat>
14393,эзэмшлээ,эзэмшил <voc>
14394,улстай,улс <com>


In [2]:
import datasets
from datasets import Dataset, DatasetDict

df['prompt'] = '<s> bb: '+ df['prompt']
df['answer'] = df['answer']+'</s>'

infl_dataset = Dataset.from_pandas(df)
infl_dataset

Dataset({
    features: ['prompt', 'answer'],
    num_rows: 14396
})

In [3]:
ds_train_devtest = infl_dataset.train_test_split(test_size=0.025, seed = 42)
ds_devtest = ds_train_devtest['test'].train_test_split(test_size=0.5, seed = 42)

ds_splits = DatasetDict({
    'train': ds_train_devtest['train'],
    'valid': ds_devtest['train'],
    'test': ds_devtest['test']
})

ds_splits

DatasetDict({
    train: Dataset({
        features: ['prompt', 'answer'],
        num_rows: 14036
    })
    valid: Dataset({
        features: ['prompt', 'answer'],
        num_rows: 180
    })
    test: Dataset({
        features: ['prompt', 'answer'],
        num_rows: 180
    })
})

In [4]:
def concatenate_columns(example):
    example["prompt"] = example["prompt"] + " " + example["answer"]
    return example

ds_splits["train"] = ds_splits["train"].map(concatenate_columns)
ds_splits["valid"] = ds_splits["valid"].map(concatenate_columns)

Map:   0%|          | 0/14036 [00:00<?, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

In [5]:
ds_splits["train"][0:10]

{'prompt': ['<s> bb: хавчаарт хавчаар <dat></s>',
  '<s> bb: санхүүд санхүү <dat></s>',
  '<s> bb: зохицлын зохицол <gen></s>',
  '<s> bb: сейфээ сейф <voc></s>',
  '<s> bb: уулаа уул <voc></s>',
  '<s> bb: эхнэртэй эхнэр <com></s>',
  '<s> bb: ордноос ордон <abl></s>',
  '<s> bb: оноонд оноо <dat></s>',
  '<s> bb: нөөцийг нөөц <acc></s>',
  '<s> bb: цогцсыг цогцос <acc></s>'],
 'answer': ['хавчаар <dat></s>',
  'санхүү <dat></s>',
  'зохицол <gen></s>',
  'сейф <voc></s>',
  'уул <voc></s>',
  'эхнэр <com></s>',
  'ордон <abl></s>',
  'оноо <dat></s>',
  'нөөц <acc></s>',
  'цогцос <acc></s>']}

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bayartsogt/mongolian-gpt2")

tokenizer_config.json:   0%|          | 0.00/207 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.33M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.10M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

In [7]:
ds_splits = ds_splits.flatten()
ds_splits["train"][0]

{'prompt': '<s> bb: хавчаарт хавчаар <dat></s>', 'answer': 'хавчаар <dat></s>'}

In [8]:
def preprocess_function(examples, tokenizer = tokenizer):
    return tokenizer(examples["prompt"])

tokenized_ds = ds_splits.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=ds_splits["train"].column_names,
)

Map (num_proc=4):   0%|          | 0/14036 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

In [9]:
block_size = 64



lm_dataset = tokenized_ds.map(group_texts, batched=True, num_proc=4)

Map (num_proc=4):   0%|          | 0/14036 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/180 [00:00<?, ? examples/s]

In [10]:
from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [11]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained("bayartsogt/mongolian-gpt2")

config.json:   0%|          | 0.00/864 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/510M [00:00<?, ?B/s]

In [12]:
training_args = TrainingArguments(
    output_dir="my_awesome_gpt-inflection-model",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["test"],
    data_collator=data_collator,
)

trainer.train()

/home/razydave/anaconda3/envs/pt_env/lib/python3.11/site-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


  0%|          | 0/969 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

{'eval_loss': 5.545279502868652, 'eval_runtime': 0.0667, 'eval_samples_per_second': 239.934, 'eval_steps_per_second': 29.992, 'epoch': 1.0}
{'loss': 2.6137, 'grad_norm': 1.3151301145553589, 'learning_rate': 9.680082559339526e-06, 'epoch': 1.55}


  0%|          | 0/2 [00:00<?, ?it/s]

{'eval_loss': 5.652219295501709, 'eval_runtime': 0.054, 'eval_samples_per_second': 296.547, 'eval_steps_per_second': 37.068, 'epoch': 2.0}


  0%|          | 0/2 [00:00<?, ?it/s]

{'eval_loss': 5.753127098083496, 'eval_runtime': 0.0493, 'eval_samples_per_second': 324.593, 'eval_steps_per_second': 40.574, 'epoch': 3.0}
{'train_runtime': 111.0243, 'train_samples_per_second': 69.769, 'train_steps_per_second': 8.728, 'train_loss': 1.9840989245718847, 'epoch': 3.0}


TrainOutput(global_step=969, training_loss=1.9840989245718847, metrics={'train_runtime': 111.0243, 'train_samples_per_second': 69.769, 'train_steps_per_second': 8.728, 'train_loss': 1.9840989245718847, 'epoch': 3.0})

In [13]:
import math

eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

  0%|          | 0/2 [00:00<?, ?it/s]

Perplexity: 315.17


In [14]:
prompt = "<s> bb: сургуулийн"

from transformers import pipeline

generator = pipeline("text-generation", model=model, tokenizer = tokenizer, num_beams=5)

ans = generator(prompt, max_length = 20)

print(str(ans[0]['generated_text']))
print(str(ans))

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


<s> bb: сургуулийн сургууль <gen> bb: төлгийг төлгө <
[{'generated_text': '<s> bb: сургуулийн сургууль <gen> bb: төлгийг төлгө <'}]


In [15]:
prompt = "<s> bb: утааны"
ans = generator(prompt, max_length = 15)
print(str(ans))

[{'generated_text': '<s> bb: утааны утаа <gen> bb:'}]
